# 적용한 내용
기존 페르소나 사전은 json으로 바로 활용 가능한 형태였다.
따라서 기존 데이터를 그대로 personas.json에 저장한 후,
[페르소나 사전 생성](#페르소나-사전-생성) 단계에서 json.load를 통해 읽어 오도록 수정했다.

# 실행 환경 감지

In [1]:
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


# API 키 로드

In [2]:
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


# 패키지 임포트

In [3]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import json

print("✅ import 완료")

✅ import 완료


# llm, prompt, chain 생성 및 작동 확인

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}")
])

chain = prompt | llm   # LCEL 파이프 연산자 (2교시 3부)

# 동작 확인
result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로."})
print(result.content)

이상치는 데이터 집합에서 다른 값들과 현저히 차이나는 값으로, 일반적인 패턴이나 분포에서 벗어난 데이터를 의미합니다.


# 페르소나 사전 생성

In [5]:
# 페르소나 사전: 이름 → system 메시지
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


# 챗봇 함수 정의

In [6]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    system_message = PERSONAS[persona_name]

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [7]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 챗봇 실행 함수 정의

In [8]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

# 챗봇 실행

In [ ]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

✅ 페르소나가 '디자이너'(으)로 변경되었습니다.
[디자이너] 페르소나는 사용자의 특성과 행동을 대표하는 가상의 인물입니다. 마치 영화의 캐릭터처럼, 특정 목표와 필요를 가진 사람을 상징합니다. UX 디자인에서 페르소나는 사용자의 요구를 이해하고, 그에 맞춘 제품이나 서비스를 설계하는 데 도움을 줍니다.

예를 들어, 한 카페의 페르소나를 생각해볼까요? '커피 애호가인 30대 직장인, 이름은 지민'이라고 설정할 수 있습니다. 지민은 매일 아침 커피를 사러 가고, 모바일 앱으로 주문하는 것을 선호합니다. 이런 페르소나를 통해 카페는 지민이 좋아할 만한 메뉴나 앱 기능을 고민하게 됩니다.

즉, 페르소나는 디자인 과정에서 사용자의 목소리를 듣고, 그들의 경험을 개선하기 위한 나침반 역할을 합니다.

✅ 페르소나가 '기획자'(으)로 변경되었습니다.
[기획자] 페르소나(Persona)는 특정 제품이나 서비스의 사용자를 대표하는 가상의 인물입니다. IT 서비스 기획에서 페르소나는 사용자 경험(UX) 디자인, 마케팅 전략, 제품 개발 등 다양한 분야에서 중요한 역할을 합니다. 페르소나는 다음과 같은 요소를 포함하여 사용자의 특성을 구체화합니다:

1. **인구통계학적 정보**: 나이, 성별, 직업, 교육 수준 등
2. **행동 패턴**: 사용자의 일상적인 행동, 기술 사용 습관, 구매 결정 과정 등
3. **목표와 필요**: 사용자가 제품이나 서비스를 통해 이루고자 하는 목표와 그에 따른 필요
4. **고민과 문제점**: 사용자가 직면하고 있는 문제나 불편함

페르소나를 정의함으로써, IT 서비스 기획자는 사용자 중심의 접근 방식을 취할 수 있으며, 사용자 가치를 극대화하는 데 필요한 기능이나 서비스를 우선순위에 따라 개발할 수 있습니다. 이를 통해 최종 사용자에게 더 나은 경험을 제공하고, 제품이나 서비스의 성공 가능성을 높일 수 있습니다.

👋 종료합니다.


: 